In [1]:
# ═══════════════════════════════════════════════════════════════
#  TRPG 调查员助手 —— 主流程 Notebook
# ═══════════════════════════════════════════════════════════════

import sys
import json
from datetime import datetime
from IPython.display import HTML, display

# 将 src/ 加入路径以导入依赖模块
sys.path.insert(0, "../src")

from scenario_core import DirectedGraph, ScenarioWorld
from llm import call_deepseek, set_llm_log_file
from prompts import build_narrative_prompt, set_prompt_log_file, parse_narrative_output
from game_loop import handle_user_input
from trpg_display import (
    display_narrative, display_scene, display_system, display_debug,
    display_input_area, render_scene_to_html, display_split_result,
)

# ── COC 7th 车卡系统（替代旧 Player 类）──
# 调查员通过前端 character.html 创建并导出 JSON，这里负责加载。
# 如果还没有创建角色卡，运行时会自动生成一个默认调查员。
from investigator import Investigator, load_investigator
from investigator.rules import roll_stats, calc_derived, create_skill_list

In [2]:
# ═══════════════════════════════════════════════════════════════
#  Prompt 日志配置
# ═══════════════════════════════════════════════════════════════

PROMPT_LOG_FILE = f"../logs/prompt_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
set_prompt_log_file(PROMPT_LOG_FILE)
set_llm_log_file(PROMPT_LOG_FILE)

In [ ]:
def run_game(character_path: str = None):
    """
    启动 TRPG 游戏主循环。

    参数:
        character_path: 调查员 JSON 文件路径（可选）。
                       如果为 None，自动查找 ../investigator/test_character.json；
                       如果文件不存在，使用默认掷骰生成的调查员。
    """
    import json as _json
    import os as _os

    # ── 从文件直接加载模组数据 ──
    with open("../data/output/scene_output_resolved_revised.json", "r", encoding="utf-8") as f:
        scenes = _json.load(f)
    with open("../data/output/res_event_resolved_revised.json", "r", encoding="utf-8") as f:
        events = _json.load(f)
    with open("../data/abstract.txt", "r", encoding="utf-8") as f:
        abstract = f.read()

    # ── 构建世界 ──
    graph = DirectedGraph(scenes=scenes, events=events)
    world = ScenarioWorld(graph, start_node="6号车厢",
                          background_story=abstract)

    # ── 加载调查员（COC 7th 车卡系统）──
    # 优先从 JSON 文件加载（前端 character.html 导出）；
    # 如果文件不存在，自动掷骰生成一个默认调查员。
    if character_path is None:
        character_path = "../investigator/test_character.json"

    if _os.path.exists(character_path):
        # ── 方式 1：从 JSON 文件加载 ──
        investigator = load_investigator(character_path)
        display_system(
            f"已加载调查员：{investigator.name} | "
            f"职业：{investigator.occupation.name if investigator.occupation else '无'} | "
            f"HP={investigator.derived.HP} SAN={investigator.derived.SAN}",
            "info"
        )
    else:
        # ── 方式 2：掷骰生成默认调查员（无 JSON 时的 fallback）──
        display_system(
            f"未找到角色卡文件 {character_path}，正在掷骰生成默认调查员...",
            "warn"
        )
        investigator = Investigator(name="调查员A", age=25, gender="男")
        investigator.stats = roll_stats()
        investigator.skills = create_skill_list()
        investigator.derived = calc_derived(investigator.stats, investigator.age)
        display_system(
            f"已生成调查员：{investigator.name} | "
            f"HP={investigator.derived.HP} SAN={investigator.derived.SAN}",
            "info"
        )

    # 注入世界（Investigator 完全替代旧 Player 类）
    world.set_player(investigator)

    # ── 确保存档目录存在 ──
    _os.makedirs("../data/saves", exist_ok=True)

    turn = 0

    # ── 开场 ──
    display_system("游戏开始。输入 /help 查看可用命令。", "info")
    display(HTML(render_scene_to_html(world)))

    try:
        initial_narrative = call_deepseek(
            build_narrative_prompt(
                world,
                user_input="（游戏开始）",
                action_result="（从沉睡中醒来，环顾四周）",
                events_result="",
            ),
            json_mode=False
        )
        init_brief, init_narr = parse_narrative_output(initial_narrative)
        display_split_result(init_brief, init_narr)
    except Exception as e:
        display_system(f"初始叙事生成失败（API 可能未配置）：{e}", "warn")

    while True:
        turn += 1
        display_input_area(turn, world.current_location)
        cmd = input().strip()
        if not cmd:
            continue

        # ── 退出 ──
        if cmd.lower() in ("exit", "quit"):
            display_system("是否保存调查员最终状态？（y/n，默认 y）", "info")
            choice = input().strip().lower()
            if choice in ("", "y", "yes"):
                final_path = "../investigator/character_final.json"
                try:
                    investigator.save(final_path)
                    display_system(f"调查员最终状态已保存至：{final_path}", "info")
                except Exception as e:
                    display_system(f"调查员保存失败：{e}", "warn")
            else:
                display_system("调查员未保存。", "info")
            display_system("游戏结束。", "warn")
            break

        # ── 帮助 ──
        if cmd.lower() == "/help":
            display_system(
                "命令列表：\n"
                "  /scene      — 查看当前场景完整信息\n"
                "  /info       — 查看结构化 JSON 状态\n"
                "  /events     — 查看已触发事件\n"
                "  /flags      — 查看世界标记\n"
                "  /char       — 查看当前调查员角色卡\n"
                "  /do 动作名   — 直接执行交互（跳过 LLM）\n"
                "  /trigger E1 — 手动触发事件\n"
                "  /save 槽位名 — 临时存档（如 /save 01）\n"
                "  /load 槽位名 — 读档（如 /load 01）\n"
                "  /charsave   — 手动保存调查员长期存档\n"
                "  /charload   — 手动加载调查员长期存档\n"
                "  直接输入     — 正常游戏（LLM 调用链）\n"
                "  exit/quit   — 退出（可选保存调查员）",
                "info"
            )
            continue

        # ── 调试命令 ──
        if cmd.lower() == "/scene":
            display(HTML(render_scene_to_html(world)))
            continue
        if cmd.lower() == "/info":
            display_debug(_json.dumps(world.get_scene_info(), ensure_ascii=False, indent=2))
            continue
        if cmd.lower() == "/events":
            active = world.get_active_event_effects()
            if active:
                for name, impact in active:
                    display_system(f"◆ {name}\n{impact}", "event")
            else:
                display_system("尚无事件触发。", "info")
            continue
        if cmd.lower() == "/flags":
            if world.flags:
                items = "\n".join(f"  {k} = {v}" for k, v in world.flags.items())
                display_system(f"世界标记：\n{items}", "info")
            else:
                display_system("世界标记：（空）", "info")
            continue
        if cmd.lower() == "/char":
            # ── 查看调查员角色卡（新增命令）──
            inv = world.player
            if not inv:
                display_system("尚未设置调查员。", "warn")
            else:
                occ_name = inv.occupation.name if inv.occupation else "无"
                stat_lines = [
                    f"STR={inv.stats.STR} CON={inv.stats.CON} SIZ={inv.stats.SIZ}",
                    f"DEX={inv.stats.DEX} APP={inv.stats.APP} INT={inv.stats.INT}",
                    f"POW={inv.stats.POW} EDU={inv.stats.EDU} LUCK={inv.stats.LUCK}",
                ]
                derived_lines = [
                    f"HP={inv.derived.HP} MP={inv.derived.MP} SAN={inv.derived.SAN}/{inv.derived.SAN_MAX}",
                    f"MOV={inv.derived.MOV} DB={inv.derived.DB} BUILD={inv.derived.BUILD} DODGE={inv.derived.DODGE}",
                ]
                occ_skills = [s for s in inv.skills if s.is_occupation]
                non_occ_boosted = [s for s in inv.skills if not s.is_occupation and s.value > s.base_value]
                info = (
                    f"══════ 调查员角色卡 ══════\n"
                    f"姓名：{inv.name} | 年龄：{inv.age} | 职业：{occ_name}\n"
                    f"──────────────────────────\n"
                    f"核心属性：\n  " + "\n  ".join(stat_lines) + "\n"
                    f"──────────────────────────\n"
                    f"衍生属性：\n  " + "\n  ".join(derived_lines) + "\n"
                    f"──────────────────────────\n"
                    f"职业技能 ({len(occ_skills)})："
                    + ", ".join(f"{s.name}={s.value}" for s in occ_skills) + "\n"
                    f"已加点兴趣技能 ({len(non_occ_boosted)})："
                    + (", ".join(f"{s.name}={s.value}" for s in non_occ_boosted) or "（无）") + "\n"
                    f"══════════════════════════"
                )
                display_system(info, "info")
            continue
        if cmd.lower().startswith("/trigger"):
            eid = cmd.split()[-1].strip().upper()
            result = world.trigger_event(eid)
            display_system(result.message, "event" if result.success else "warn")
            continue
        if cmd.lower().startswith("/do"):
            inter_name = cmd[3:].strip()
            result = world.execute_interaction(inter_name)
            if result.success:
                display_system(result.message, "info")
                # 消费副作用
                from game_loop import _apply_side_effects
                side_msgs = _apply_side_effects(world, result.side_effects)
                for msg in side_msgs:
                    display_system(msg, "info")
            else:
                display_system(result.message, "warn")
            continue

        # ── 存档 / 读档 ──
        if cmd.lower().startswith("/save"):
            slot = cmd[5:].strip() or "auto"
            path = f"../data/saves/save_{slot}.json"
            try:
                world.save_state(path)
                display_system(f"存档成功 → {path}", "info")
            except Exception as e:
                display_system(f"存档失败：{e}", "warn")
            continue
        if cmd.lower().startswith("/load"):
            slot = cmd[5:].strip() or "auto"
            path = f"../data/saves/save_{slot}.json"
            if not _os.path.exists(path):
                display_system(f"存档不存在：{path}", "warn")
                continue
            try:
                nonlocal_world = ScenarioWorld.load_state(path)
                # 用读档回来的世界替换当前 world
                world.graph = nonlocal_world.graph
                world.current_location = nonlocal_world.current_location
                world.triggered_events = nonlocal_world.triggered_events
                world.completed_interactions = nonlocal_world.completed_interactions
                world.flags = nonlocal_world.flags
                world.background_story = nonlocal_world.background_story
                world.memory = nonlocal_world.memory
                if nonlocal_world.player:
                    world.player = nonlocal_world.player
                display_system(f"读档成功 ← {path}", "info")
                display(HTML(render_scene_to_html(world)))
            except Exception as e:
                display_system(f"读档失败：{e}", "warn")
            continue

        # ── 调查员长期存档 / 读档 ──
        if cmd.lower().startswith("/charsave"):
            path = cmd[9:].strip() or "../investigator/character_manual.json"
            try:
                world.player.save(path)
                display_system(f"调查员已保存 → {path}", "info")
            except Exception as e:
                display_system(f"调查员保存失败：{e}", "warn")
            continue
        if cmd.lower().startswith("/charload"):
            path = cmd[9:].strip() or "../investigator/character_manual.json"
            if not _os.path.exists(path):
                display_system(f"调查员存档不存在：{path}", "warn")
                continue
            try:
                world.player = Investigator.load(path)
                display_system(
                    f"调查员已加载 ← {path}\n"
                    f"  {world.player.name} | "
                    f"HP={world.player.derived.HP} SAN={world.player.derived.SAN}",
                    "info"
                )
            except Exception as e:
                display_system(f"调查员加载失败：{e}", "warn")
            continue

        # ── 正常游戏流程 ──
        result = handle_user_input(cmd, world)
        display_split_result(result["brief"], result["narrative"])


# 启动
# 如果有 JSON 角色卡，传入路径；否则自动掷骰生成
run_game()
# run_game("../investigator/test_character.json")  # 显式指定角色卡路径